In [28]:
# 1. 라이브러리 불러오기

import os
import requests

from pathlib import Path
from dotenv import load_dotenv

In [29]:
# 2. 프로젝트 경로 설정

project_root = Path.cwd().parent

project_root

WindowsPath('c:/Users/EZ/Desktop/루키즈/1. Python/Projects/NASA_NEO_Hazard_Prediction')

In [30]:
# 3. 환경 변수 불러오기

load_dotenv(project_root / ".env")

NASA_API_KEY = os.getenv("NASA_API_KEY")

if not NASA_API_KEY:
    raise ValueError("NASA_API_KEY가 설정되지 않았습니다.")

print("NASA API Key 로드 완료")

NASA API Key 로드 완료


In [31]:
# 4. Browse API 설정

BASE_URL = "https://api.nasa.gov/neo/rest/v1/neo/browse"

In [32]:
# 5. 첫 페이지 요청

params = {
    "page": 0,
    "api_key": NASA_API_KEY
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=10
)

print("Status Code:", response.status_code)

Status Code: 200


In [33]:
# 6. HTTP 오류 확인

response.raise_for_status()

print("API 요청 성공")

API 요청 성공


In [34]:
# 7. JSON 변환

data = response.json()

type(data)

dict

In [35]:
# 8. 최상위 구조 확인

data.keys()

dict_keys(['links', 'page', 'near_earth_objects'])

In [36]:
# 9. 페이지 정보 확인

data["page"]

{'size': 20, 'total_elements': 62130, 'total_pages': 3107, 'number': 0}

In [37]:
page_info = data["page"]

print("페이지 크기:", page_info["size"])
print("전체 데이터 수:", page_info["total_elements"])
print("전체 페이지 수:", page_info["total_pages"])
print("현재 페이지:", page_info["number"])

페이지 크기: 20
전체 데이터 수: 62130
전체 페이지 수: 3107
현재 페이지: 0


In [38]:
# 10. 현재 페이지의 소행성 개수

len(data["near_earth_objects"])

20

In [39]:
# 11. 첫 번째 소행성 데이터 확인

asteroid = data["near_earth_objects"][0]

asteroid

{'links': {'self': 'http://api.nasa.gov/neo/rest/v1/neo/2000433?api_key=7dxBQ494kiYNn53Rcd6Il4UaOG2jzl56tcUuib1R'},
 'id': '2000433',
 'neo_reference_id': '2000433',
 'name': '433 Eros (A898 PA)',
 'name_limited': 'Eros',
 'designation': '433',
 'nasa_jpl_url': 'https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=2000433',
 'absolute_magnitude_h': 10.4,
 'estimated_diameter': {'kilometers': {'estimated_diameter_min': 22.1082810359,
   'estimated_diameter_max': 49.435619262},
  'meters': {'estimated_diameter_min': 22108.281035909,
   'estimated_diameter_max': 49435.619261962},
  'miles': {'estimated_diameter_min': 13.7374446956,
   'estimated_diameter_max': 30.7178601764},
  'feet': {'estimated_diameter_min': 72533.7327538517,
   'estimated_diameter_max': 162190.3570994153}},
 'is_potentially_hazardous_asteroid': False,
 'close_approach_data': [{'close_approach_date': '1900-12-27',
   'close_approach_date_full': '1900-Dec-27 01:30',
   'epoch_date_close_approach': -2177879400000,
   

# NASA NeoWs API 컬럼 정리

## 최상위 컬럼

| 컬럼명 | 설명 | 머신러닝 사용 여부 |
| :--- | :--- | :---: |
| `links` | 해당 소행성 API 정보 링크 | ❌ |
| `id` | NASA에서 부여한 소행성 고유 ID | ❌ |
| `neo_reference_id` | NEO 데이터베이스 참조 ID | ❌ |
| `name` | 소행성 이름 | ❌ |
| `nasa_jpl_url` | NASA JPL 상세 정보 페이지 URL | ❌ |
| `absolute_magnitude_h` | 소행성의 절대등급(H). 값이 작을수록 밝고 일반적으로 크기가 큰 소행성 | ✅ |
| `estimated_diameter` | 소행성의 추정 지름(단위별 제공) | ✅ |
| `is_potentially_hazardous_asteroid` | 잠재적 위험 소행성 여부(True/False) | ⭐ Target |
| `close_approach_data` | 지구 접근 정보(날짜, 속도, 거리 등) | ✅ |
| `is_sentry_object` | NASA Sentry 위험 감시 시스템 등록 여부 | △ |

---

## estimated_diameter

소행성의 추정 크기를 다양한 단위로 제공.

### kilometers

| 컬럼명 | 설명 | 사용 여부 |
| :--- | :--- | :---: |
| `estimated_diameter_min` | 추정 최소 지름(km) | ✅ |
| `estimated_diameter_max` | 추정 최대 지름(km) | ✅ |

### meters

| 컬럼명 | 설명 | 사용 여부 |
| :--- | :--- | :---: |
| `estimated_diameter_min` | 추정 최소 지름(m) | ❌ |
| `estimated_diameter_max` | 추정 최대 지름(m) | ❌ |

### miles

| 컬럼명 | 설명 | 사용 여부 |
| :--- | :--- | :---: |
| `estimated_diameter_min` | 추정 최소 지름(mi) | ❌ |
| `estimated_diameter_max` | 추정 최대 지름(mi) | ❌ |

### feet

| 컬럼명 | 설명 | 사용 여부 |
| :--- | :--- | :---: |
| `estimated_diameter_min` | 추정 최소 지름(ft) | ❌ |
| `estimated_diameter_max` | 추정 최대 지름(ft) | ❌ |

> 프로젝트에서는 **kilometers 단위만 사용**.

---

## close_approach_data

소행성이 지구에 접근하는 정보를 담고 있음.

| 컬럼명 | 설명 | 사용 여부 |
| :--- | :--- | :---: |
| `close_approach_date` | 지구 접근 날짜 | ✅ |
| `close_approach_date_full` | 지구 접근 날짜 및 시간 | △ |
| `epoch_date_close_approach` | 접근 시각(Epoch Timestamp) | ❌ |
| `relative_velocity` | 접근 속도 정보 | ✅ |
| `miss_distance` | 지구와의 최근접 거리 | ✅ |
| `orbiting_body` | 기준 천체(대부분 Earth) | ❌ |

---

## relative_velocity

소행성의 접근 속도를 다양한 단위로 제공.

| 컬럼명 | 설명 | 사용 여부 |
| :--- | :--- | :---: |
| `kilometers_per_second` | 초당 속도(km/s) | ❌ |
| `kilometers_per_hour` | 시간당 속도(km/h) | ✅ |
| `miles_per_hour` | 시간당 속도(mi/h) | ❌ |

> 프로젝트에서는 **kilometers_per_hour**를 사용.

---

## miss_distance

지구와의 최근접 거리를 다양한 단위로 제공.

| 컬럼명 | 설명 | 사용 여부 |
| :--- | :--- | :---: |
| `astronomical` | 천문단위(AU) | ❌ |
| `lunar` | 달과의 거리 기준 | ❌ |
| `kilometers` | 최근접 거리(km) | ✅ |
| `miles` | 최근접 거리(mi) | ❌ |

> 프로젝트에서는 **kilometers**를 사용.

---

# 머신러닝 학습에 사용할 Feature

| Feature | 설명 |
| :--- | :--- |
| `absolute_magnitude_h` | 절대등급(H) |
| `estimated_diameter_min` | 추정 최소 지름(km) |
| `estimated_diameter_max` | 추정 최대 지름(km) |
| `relative_velocity_kilometers_per_hour` | 지구 접근 속도(km/h) |
| `miss_distance_kilometers` | 지구와의 최근접 거리(km) |

---

# Target

| Target | 설명 |
| :--- | :--- |
| `is_potentially_hazardous_asteroid` | 잠재적 위험 소행성 여부(True / False) |

In [40]:
# 12. 주요 컬럼 확인

print("ID:", asteroid["id"])
print("Name:", asteroid["name"])
print("Absolute Magnitude H:", asteroid["absolute_magnitude_h"])

print(
    "Potentially Hazardous:",
    asteroid["is_potentially_hazardous_asteroid"]
)

print(
    "Sentry Object:",
    asteroid["is_sentry_object"]
)

ID: 2000433
Name: 433 Eros (A898 PA)
Absolute Magnitude H: 10.4
Potentially Hazardous: False
Sentry Object: False


In [ ]:
# 13. 추정 지름 확인

diameter = asteroid["estimated_diameter"]["kilometers"]

print(
    "Estimated Diameter Min (km):",
    diameter["estimated_diameter_min"]
)

print(
    "Estimated Diameter Max (km):",
    diameter["estimated_diameter_max"]
)

Estimated Diameter Min: 0.2053784995
Estimated Diameter Max: 0.459240286


In [41]:
# 14. 지구 접근 데이터 개수 확인

len(asteroid["close_approach_data"])

34

In [42]:
asteroid["close_approach_data"][:3]

[{'close_approach_date': '1900-12-27',
  'close_approach_date_full': '1900-Dec-27 01:30',
  'epoch_date_close_approach': -2177879400000,
  'relative_velocity': {'kilometers_per_second': '5.5786191875',
   'kilometers_per_hour': '20083.0290749201',
   'miles_per_hour': '12478.8132604691'},
  'miss_distance': {'astronomical': '0.3149291693',
   'lunar': '122.5074468577',
   'kilometers': '47112732.928149391',
   'miles': '29274494.7651919558'},
  'orbiting_body': 'Earth'},
 {'close_approach_date': '1907-11-05',
  'close_approach_date_full': '1907-Nov-05 03:31',
  'epoch_date_close_approach': -1961526540000,
  'relative_velocity': {'kilometers_per_second': '4.3944908885',
   'kilometers_per_hour': '15820.1671985367',
   'miles_per_hour': '9830.0366684463'},
  'miss_distance': {'astronomical': '0.4714855425',
   'lunar': '183.4078760325',
   'kilometers': '70533232.893794475',
   'miles': '43827318.620434755'},
  'orbiting_body': 'Earth'},
 {'close_approach_date': '1917-04-20',
  'close_ap